# 데이터 출처

## 아동인구수 비율
1. KOSIS 국가통계포털 -> 국내통계 -> 주제별 통계
    - https://kosis.kr/statisticsList/statisticsListIndex.do?vwcd=MT_ZTITLE&menuId=M_01_01&outLink=Y&entrType=
2. 인구 -> 주민등록인구현황 -> 행정구역(시군구)별/1세별 주민등록인구
    - https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B04006&conn_path=I2
    - 행정안전부,「주민등록인구현황」, 2025.05, 2025.06.24, 행정구역(시군구)별/1세별 주민등록인구
3. 조회설정 -> 조회조건
    - 항목: 총인구수
    - 행정구역(시군구)별: 중구, 서구, 동구, 영도구, 부산진구, 동래구, 남구, 북구, 해운대구, 사하구, 금정구, 강서구, 연제구, 수영구, 사상구, 기장군
    - 연령별: 계, 0세, 1세, 2세, 3세, 4세, 5세, 6세, 7세, 8세, 9세, 10세, 11세, 12세
        - 연령 구간을 0~12세로 선택한 이유: '부산광역시 어린이복합문화공간 조성 및 운영 지원 조례' 제2조에서 어린이를 13세 미만의 사람으로 정의하고 있습니다. kosis의 통계자료를 보면 1세 단위로 끊어서 볼 수 있어 13세 미만인 12세까지를 어린이 나이로 정했습니다.
    - 시점: 2025.05
4. 다운로드 -> 파일형태: CSV(인코딩: UTF-8) -> 다운로드
    - 파일명: 행정구역_시군구_별_1세별_주민등록인구_20250624222026.csv

# 필터링 과정
- 먼저 0~12세 인구수를 모두 합산한 후 연령별에서 추출한 '계' 데이터의 시군구별 총인구수를 나누어서 시군구별 어린이 인구수 비율을 계산했습니다.
- 부산광역시_시군구/부산광역시_시군구_데이터출처.md에서 다운받은 LARD_ADM_SECT_SGG_26_202505.shp 파일을 불러와서 시군구별 행정경계 데이터가 담긴 shp 파일과 시군구별 어린이 인구수 비율 데이터가 담긴 csv 파일을 시군구별로 병합했습니다.
- 마지막으로 합쳐진 데이터프레임을 shp 파일로 저장했습니다.

In [2]:
import pandas as pd
import geopandas as gpd

# 실제 파일 경로로 수정
bs1 = pd.read_csv('../../Data_list/source_data/행정구역_시군구_별_1세별_주민등록인구_20250624222026.csv')

# 0세~12세에 해당하는 연령 리스트 생성
ages = [f'{i}세' for i in range(13)]

# 0~12세 인구수: 연령별이 0세~12세인 행만 필터링 후 시군구별 합계
child = bs1[bs1['연령별'].isin(ages)]
child['총인구수 (명)'] = pd.to_numeric(child['총인구수 (명)'], errors='coerce')
child_sum = child.groupby('행정구역(시군구)별')['총인구수 (명)'].sum()

# 전체 인구수: 연령별이 '계'인 행만 필터링 후 시군구별 합계
total = bs1[bs1['연령별'] == '계']
total['총인구수 (명)'] = pd.to_numeric(total['총인구수 (명)'], errors='coerce')
total_sum = total.set_index('행정구역(시군구)별')['총인구수 (명)']

# 시군구별 0~12세 비율 계산
ratio = (child_sum / total_sum).reset_index()
ratio.columns = ['행정구역(시군구)별', '0~12세_비율']

# SHP 파일(행정경계) 읽기 (원본을 수정하지 않기 위해 복사본 생성)
gdf_orig = gpd.read_file('../../Data_list/source_data/LARD_ADM_SECT_SGG_부산/LARD_ADM_SECT_SGG_26_202505.shp')
gdf = gdf_orig.copy()  # 복사본 생성

# SHP 파일의 'SGG_NM'에서 '부산광역시' 문자열 제거해서 시군구명만 추출
gdf['시군구명'] = gdf['SGG_NM'].str.replace('부산광역시 ', '', regex=False)

# 시군구명을 기준으로 CSV와 SHP를 병합(merge)
gdf = gdf.merge(ratio, left_on='시군구명', right_on='행정구역(시군구)별', how='left')
gdf = gdf.drop(columns=['시군구명', '행정구역(시군구)별'])
gdf.to_file('../../Data_list/preprocessing_result/아동인구수전처리결과/시군구별_어린이_비율.shp', encoding='cp949')

# 5. 결과 확인
print(gdf)

   ADM_SECT_C      SGG_NM  SGG_OID COL_ADM_SE  \
0       26110    부산광역시 중구   1009.0      26110   
1       26140    부산광역시 서구    369.0      26140   
2       26170    부산광역시 동구    690.0      26170   
3       26200   부산광역시 영도구     35.0      26200   
4       26230  부산광역시 부산진구    689.0      26230   
5       26260   부산광역시 동래구     49.0      26260   
6       26290    부산광역시 남구      NaN      26290   
7       26320    부산광역시 북구     49.0      26320   
8       26350  부산광역시 해운대구   1329.0      26350   
9       26380   부산광역시 사하구   1009.0      26380   
10      26410   부산광역시 금정구    369.0      26410   
11      26440   부산광역시 강서구     51.0      26440   
12      26470   부산광역시 연제구    369.0      26470   
13      26500   부산광역시 수영구    369.0      26500   
14      26530   부산광역시 사상구     49.0      26530   
15      26710   부산광역시 기장군   1329.0      26710   

                                             geometry  0~12세_비율  
0   POLYGON ((385270.012 281905.258, 385273.98 281...  0.029072  
1   MULTIPOLYGON (((383830.943 274

C:\Users\tjral\AppData\Local\Temp\ipykernel_18256\992482586.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  child['총인구수 (명)'] = pd.to_numeric(child['총인구수 (명)'], errors='coerce')
C:\Users\tjral\AppData\Local\Temp\ipykernel_18256\992482586.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  total['총인구수 (명)'] = pd.to_numeric(total['총인구수 (명)'], errors='coerce')
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Could not decode error message to UTF-8. Raw error: b"Normalized/la